In [ ]:
# --- system setup ---
import sys
import os
sys.path.append(os.path.abspath(".."))

In [ ]:
from ib_insync import Stock, util
from ibkr.Class_IBKR_IB import IBKR_IB
ibkr = IBKR_IB(port=7496)

async def start_ibkr():
    await ibkr.connect()
    print("IBKR connected:", ibkr.ib.isConnected())

await start_ibkr()

In [ ]:
lookback_period = "1 D"
length_of_each_period = "1 day"
use_regular_trading_hours = True

prices_to_use = "MIDPOINT"  #TRADES FOR BTC FUTURES

'''

    "TRADES"
    "MIDPOINT"
    "BID"
    "ASK"
    "BID_ASK"
    "ADJUSTED_LAST"
    "HISTORICAL_VOLATILITY"
    "OPTION_IMPLIED_VOLATILITY"
    "FEE_RATE"
    "REBATE_RATE"
    "SCHEDULE"
        
'''

In [ ]:
symbol_list = ['ARKB',
                'BITB',
                'BRRR',
                'BTC',
                'BTCW',
                'EZBC',
                'FBTC',
                'GBTC',
                'HODL',
                'IBIT']


In [ ]:
dict_ = {}

for sym in symbol_list:

    contract = Stock(sym, 'SMART', 'USD')
    await ibkr.ib.qualifyContractsAsync(contract)

    bars = await ibkr.ib.reqHistoricalDataAsync(
                contract=contract,
                endDateTime="",
                durationStr=lookback_period,
                barSizeSetting=length_of_each_period,
                whatToShow=prices_to_use,
                useRTH=use_regular_trading_hours,
                formatDate=1,
                keepUpToDate=False
            )

    df = util.df(bars)
    df['symbol'] = sym
    dict_[sym] = df

In [ ]:
import pandas as pd

dfs = [df for df in dict_.values()]

df_all = pd.concat(dfs, ignore_index=True)

df_all.drop(columns=['open', 'high', 'low', 'volume', 'average', 'barCount'], inplace=True)

df_all['time'] = df_all['date'].dt.time
df_all['date'] = df_all['date'].dt.date

df_all